> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 历史学习课：用于理解问题，不取代当前H01/H02。窗口对比只针对已开放的三期，不要用它探索待封存的最终标签。

# 第2课·完整推导版：从业务用途到窗口定义

这份 notebook 补充旧第2课，不覆盖你的旧文件或笔记。逐段顺序是：问题 → 操作 → 实际结果 → 解释 → 决定。
本课没有训练模型；所有人数与比例均为标签的描述统计，不能当作模型成绩。

## 0. 我们究竟在选择什么？

| 名称 | 含义 | 本课例子 |
|---|---|---|
| 快照日 t | 假装站在哪一天预测 | 2018-01-01 |
| 历史窗口 H | 向过去回看多长 | 六个日历月 |
| 未来窗口 P | 预测之后多长时间内的事件 | 90天 |
| 评分频率 | 多久生成一次名单 | 可以每月一次 |
| 训练/验证/测试 | 哪批客户快照用于学参数、选方案、评估 | 不是把某个月订单直接当测试集 |

H 与 P 不需要相等，评分频率也不需要等于P。六个日历月不是固定180天，90天也不是总等于三个日历月。

## 1. 提出业务用途——先写一个明确假设

没有真实业务方，所以我们先写课程练习场景，而不是声称这是Olist实际经营需求：
“运营人员每月需要给近半年购买过的客户生成优先级名单，先研究其下一季自然复购倾向，再考虑如何验证联系效果。”

下一季使90天成为合理候选，但没有证明它最佳。若改成一个月内评价效果，应优先研究30天。
30/60/90/120/180天用于检查期限的可观察性与代价，不用来凭正样本多寡挑最好看的任务。
联系名单的效果与预测能力是不同问题；本数据没有营销干预随机实验。

## 2. 先读全部原始时间记录，再解释筛选
数据来自小组仓库原始包，Olist 数据许可 CC BY-NC-SA 4.0。
https://github.com/CercaTrovato/olist-repurchase-analysis


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

DATA = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/olist-v2.zip').is_file()), Path.cwd()) / 'data' / 'olist-v2.zip'
assert DATA.exists(), '请在 olist_learning 文件夹运行，确保旁边有 data/olist-v2.zip'
def read(name):
    with ZipFile(DATA) as z:
        with z.open(name) as f:
            return pd.read_csv(f)

raw_orders = read('olist_orders_dataset.csv')
items = read('olist_order_items_dataset.csv')
customers = read('olist_customers_dataset.csv')
raw_orders['purchase_at'] = pd.to_datetime(raw_orders['order_purchase_timestamp'])
raw_orders['approved_at'] = pd.to_datetime(raw_orders['order_approved_at'])

# 一单多条明细，先把商品金额合成每单一行。
amounts = items.groupby('order_id')['price'].sum().rename('amount')
orders = raw_orders.merge(amounts, on='order_id', how='left', validate='one_to_one')
orders = orders.merge(customers[['customer_id', 'customer_unique_id']],
                      on='customer_id', validate='one_to_one')

# 暂用订单资格：已批准且商品金额为正；不按后来的最终状态筛选。
eligible = orders.loc[orders['approved_at'].notna() & orders['amount'].gt(0)].copy()
print('原始订单数：', len(raw_orders))
print('有批准时间：', raw_orders['approved_at'].notna().sum())
print('合格订单数：', len(eligible))
print('原始下单时间：', raw_orders['purchase_at'].min(), '至', raw_orders['purchase_at'].max())
print('批准时间：', raw_orders['approved_at'].min(), '至', raw_orders['approved_at'].max())


**怎样解释？** 原始下单时间覆盖2016-09至2018-10，批准时间结束于2018-09。两列表示不同事件。
我们可以选择按下单预测，但那就要另写标签规则；本练习沿用“批准订单”，所以使用批准时间作为事件时间。
“合格”不是要求一个客户买很多次：客户过去H内至少有一笔这样的订单就进入名单。


In [ ]:
months = pd.period_range(raw_orders['purchase_at'].min().to_period('M'),
                         raw_orders['purchase_at'].max().to_period('M'), freq='M')
monthly = pd.DataFrame({
    '原始下单数_按下单月份': raw_orders.groupby(raw_orders['purchase_at'].dt.to_period('M')).size(),
    '有批准时间的订单_按批准月份': raw_orders.groupby(raw_orders['approved_at'].dt.to_period('M')).size(),
    '合格订单_按批准月份': eligible.groupby(eligible['approved_at'].dt.to_period('M')).size()
}).reindex(months).fillna(0).astype(int)
print(monthly.to_string())


**怎样解释整张表？**

- 例如2018-01的7140，是按批准时间落在1月统计的合格订单数，不是客户数、不是复购人数。
- 同一订单可能12月下单、1月批准，因此三列月份分布不同。
- 2016-11在文件中没有记录，2016-12只有1笔；应当检查来源，不能断言当时平台没有业务。
- 2018-09有16笔下单、1笔批准且合格的订单；10月有4笔下单、没有批准订单。不能把原始下单日期上界直接用作批准订单完整上界。
- 输出中的0只表示当前文件没有观察到该类记录，不等于证明现实中没有订单。

**工作决定：** 暂取2017-01-01作为我们愿意使用的历史起点，2018-09-01作为不含当天的观察上界。它们是依据边界稀疏现象提出的研究假设，不是数据完整性认证。继续保留全部原始数据，按研究窗口取子集，不删除原文件中的边界记录。

验证集不是“挑9月订单”。即使9月有很多历史客户可评分，缺少之后完整结果，也无法进行完整的90天分类评价。这里缺的是未来标签，不能用均值、众数或模型预测来填成真实答案。


In [ ]:
HISTORY_START = pd.Timestamp('2017-01-01')
OBSERVATION_END = pd.Timestamp('2018-09-01')  # 不含当天
print('工作版历史起点：', HISTORY_START.date())
print('工作版观察上界：', OBSERVATION_END.date())


## 3. 历史为什么暂用六个月？也要有意识地选择
先固定2018-01-01预测未来90天，检查回看3/6/12个月时纳入多少买家。注意：本课把客户资格也与H绑定，因此改变H会同时改变名单，不是同一人群上的纯特征实验。


In [ ]:
def history_ids(cut, history_months):
    start = cut - pd.DateOffset(months=history_months)
    hist = eligible.loc[eligible['approved_at'].ge(start) & eligible['approved_at'].lt(cut)]
    return pd.Index(hist['customer_unique_id'].unique())

cut = pd.Timestamp('2018-01-01')
future90 = eligible.loc[eligible['approved_at'].ge(cut)
                        & eligible['approved_at'].lt(cut + pd.Timedelta(days=90)), 'customer_unique_id']
history_rows=[]
for h in [3, 6, 12]:
    ids = history_ids(cut, h)
    n = int(ids.isin(future90).sum())
    history_rows.append({'历史月数': h, '客户数': len(ids), '90天复购人数': n,
                         '90天复购比例': n / len(ids)})
print(pd.DataFrame(history_rows).to_string(index=False))


**结果解释与决定：** 3个月17297人，6个月29368人，12个月43155人；对应90天复购人数136、230、314。
12个月名单更大，复购人数更多，但比例并没有上升。这不证明12个月坏，也不证明6个月最优。
本场景想关注“近半年有购买行为”的已有客户，所以把6个月作为工作版资格与特征窗口，兼顾近期性和覆盖量。若要比较不同特征回看长度，应该先固定客户资格，再单独改变特征窗口。

## 4. 固定这批客户，仅改变未来窗口
1月1日的六个月历史为[2017-07-01, 2018-01-01)。这批29368名买家在每个期限比较中保持相同；不是只有这些月份的订单才能用于整个项目。
以下只是计算真实标签，尚未让模型做任何预测。


In [ ]:
def horizon_table(snapshot, horizons=(30, 60, 90, 120, 180), history_months=6):
    t = pd.Timestamp(snapshot)
    start = t - pd.DateOffset(months=history_months)
    ids = history_ids(t, history_months)
    rows=[]
    for p in horizons:
        end = t + pd.Timedelta(days=p)
        covered = start >= HISTORY_START and end <= OBSERVATION_END
        if covered:
            future = eligible.loc[eligible['approved_at'].ge(t)
                                  & eligible['approved_at'].lt(end), 'customer_unique_id']
            n = int(ids.isin(future).sum())
        else:
            n = None
        rows.append({'快照日': str(t.date()), '未来天数': p, '客户数': len(ids),
                     '未来窗口结束_不含': str(end.date()), '在工作范围内': covered,
                     '复购人数': n, '复购比例': n / len(ids) if n is not None and len(ids) else None})
    return pd.DataFrame(rows)

jan = horizon_table('2018-01-01')
print(jan.to_string(index=False))
assert jan['复购人数'].is_monotonic_increasing


**逐项解释：**

| 期限 | 复购人数 | 读法 |
|---|---:|---|
| 30天 | 99 | 反馈快，但可用正样本更少 |
| 60天 | 164 | 比30天多65人，等待也更长 |
| 90天 | 230 | 比60天多66人，与约一季场景相符 |
| 120天 | 284 | 比90天多54人，可作为邻近期限敏感性方案 |
| 180天 | 401 | 结果更多，但要等约半年，时点安排受限 |

同一人群、嵌套未来窗口下，复购人数不会随期限变长而减少，这是标签定义的性质，不是预测准确性。
99与230都不是命中人数：这里根本没有模型输出。不能据此计算模型Precision、Recall、AP或ROC-AUC。
“信息更多”不等于有统计充分性保证，没有通用的200人合格线。

## 5. 换时点检查：是不是只有1月如此？


In [ ]:
tables = [horizon_table(s) for s in ['2017-10-01', '2018-01-01', '2018-04-01']]
sensitivity = pd.concat(tables, ignore_index=True)
print(sensitivity.to_string(index=False))


**怎样解释？** 90天在三个快照的复购比例约为1.00%、0.78%、0.79%，人数211、230、299。
它表明多个时期都能形成稀少但非零的正样本，且比例随时期变化；不能证明模型稳定，也不能证明90天最好。
4月快照的180天窗口超出工作上界，所以不输出貌似完整的复购率。

同一买家可在10月和1月各有一行客户快照，这是定期评分场景允许的。历史窗口重叠、客户重复不自动构成泄漏，但评估结果有关联，不能把三期简单当成三个独立实验。同一行客户快照的X必须只用当时已知的信息。

严格未触碰测试强调“没有根据它的标签、表现反复选择方案”。检查数据结构本身不自动使测试无效。本项目这些时期的结果已被查看并用于探索，因此应明确报告探索性质；数据有限时可冻结规则后做时间回测，但不要包装成全新独立确认。

## 6. 更长期限怎样压缩验证空间？实际排列一次
下面限定：H=6个月；历史不早于2017-01；快照按月初候选；标签结束不晚于2018-09-01。
“月初候选数”允许未来窗口相互重叠；“不重叠安排”是从最早可用月初开始贪心挑选标签窗口不重叠的日期，用于说明时间预算，不是统计独立性保证。


In [ ]:
candidate_cuts = pd.to_datetime(['2017-10-01', '2018-01-01', '2018-04-01'])
schedule_rows=[]
for p in [30, 60, 90, 120, 180]:
    feasible=[t for t in candidate_cuts if t + pd.Timedelta(days=p) <= OBSERVATION_END]
    spaced=[]
    for t in feasible:
        if not spaced or t >= spaced[-1] + pd.Timedelta(days=p):
            spaced.append(t)
    schedule_rows.append({'未来天数': p, '可用月初数_可重叠': len(feasible),
                          '结果不重叠安排数': len(spaced),
                          '不重叠快照例子': ', '.join(str(t.date()) for t in spaced)})
print(pd.DataFrame(schedule_rows).to_string(index=False))


**具体取舍：** 在这组限制下，90天可安排4个不重叠结果窗口，120天3个，180天2个。
例如90天可用2017-07、2017-10、2018-01、2018-04月初。180天只能安排2017-07、2018-01月初。
这让90天比180天更方便安排训练、验证和后期评价，但120天也能安排3期，所以不是90天唯一可行的证明。
若允许标签重叠，可以有更多快照；仍需保证预测当时训练标签已经可知，并考虑相关性。不能为增加行数偷用未来标签。
评分频率可以每月一次，即使P=90天；每月评分与每期标签完全不重叠是两个不同要求。

## 7. 确定工作版期限：写下决定，不假装发现唯一答案

**本练习决定：H=6个月，P=90天。**

理由有四条：①明确采用约一季自然复购倾向的课程场景；②在选定时点，历史与未来窗口都在工作观察范围内；③多个时点能构造正样本，但比例低，要谨慎评价；④相比180天，能保留更多不重叠时间阶段。60与120天仍是可行替代，不声称90天最好。

如果业务明确改成一个月内评价，应重设P=30天；如果来源说明证明边界月份不完整，需要调整时点；若需要预测首次购买客户，应修改纳入规则。改变问题时，标签、样本和说明一并更新。

下一阶段才在固定90天任务下训练baseline与分类模型，回答“谁预测得更好”。如果将来要比较期限的业务收益，需要统一成本、行动、预算和结果评价期限；不能直接拿不同标签的指标挑任务。

## 8. 业务口径：把自然语言变成别人能复算的规则

“业务口径”就是统计/预测时约定的对象、事件、时间与金额等定义。不是行业黑话，也不只是字段名。

| 项目 | 工作版规则 |
|---|---|
| 服务谁 | 客户运营人员；课程假设 |
| 用途 | 排序已有客户的下一季自然复购倾向；不声称营销新增效果 |
| 买家标识 | customer_unique_id；customer_id用于关联 |
| 合格订单 | order_approved_at非空，整单price之和>0 |
| 订单事件时间 | 批准时间；不把下单与批准混用 |
| 纳入客户 | [t-6个月,t)至少一笔合格订单 |
| 历史输入 | 截至t已知的记录；不含未来订单或未来最终状态 |
| 复购标签 | [t,t+90天)至少一笔合格新订单记1，完整可观察但无订单记0 |
| 观察不足 | 不把未观察的未来补成0；换早期快照、缩短期限，或另用删失方法 |
| 金额M | 历史订单商品price合计，不含运费，不是利润 |
| 边界 | 左闭右开，恰好在t的订单算未来，不算历史 |
| 已知限制 | 数据边界是工作假设；正样本稀少；各期结果已探索 |

例子：某订单12月31日下单、1月2日批准。若t=1月1日且按批准时间定义，它属于未来订单，不能提前进入历史。
若有人改成按下单日统计，该订单会被放进历史，两份数据就不可直接混用。这就是统一口径的价值。

## 9. 一个客户怎样进入三个时间集合？
可用教学安排：2017-10-01训练快照、2018-01-01验证快照、2018-04-01后期评价快照；目前全部属于已看过的探索数据。
每份集合都是“客户快照的历史特征X + 此后90天标签y”，不是三个单月的原始订单。
训练快照未来标签在2017-12-30前结束，1月验证时这些训练结果已可观察；验证标签在4月1日前结束，再评价4月快照。

这不要求H=P，也不要求所有实务评分时点相隔90天。只是本练习选择易解释的时间安排。

## 10. 修正后的自测答案
1. 30变90天复购人数增加不代表模型更准；没有训练或预测。
2. 若明确一个月内评价，优先研究30天；不与一个日历月完全等同。
3. 4月快照180天观察不足，不能把没看到订单者全部标0，也不能用普通缺失值填补生成真实未来。可用早期快照、较短期限，或另学删失/生存分析；已观察到复购的人可知为正，但剩余者不能完整判负。
4. 接受“90天符合所设场景且有研究可行性，但未证明优于60/120天”，因为期限定义研究问题，最优必须有明确业务目标及比较证据；本课只检查标签与时间可行性。

本课完成标准：能解释t、H、P；知道每张输出表是什么分母；能说明90天选择的假设、证据和局限。完成后继续订单表关联与核查，不需要等到证明90天最优。
